#⚽ Projeto Data Science: Histórico do Campeonato Brasileiro
## Integração com BigQuery e análise exploratória de dados (EDA) em Python
---
**Objetivo:** Acessar a infraestrutura de dados armazenada no Google Cloud (Storage e BigQuery) utilizando a biblioteca `google-cloud-bigquery` para extrair e analisar o histórico de partidas do Campeonato Brasileiro.

**Fontes de Dados (Kaggle):**
* [Histórico do Campeonato Brasileiro (2003-2024)](https://www.kaggle.com/datasets/lucasyukioimafuko/brasileirao-serie-a-2006-2022)
* [Histórico de Partidas e média de público.](https://www.kaggle.com/datasets/rczekster/matches-brazilian-football-from-2003-to-2019)

In [ ]:
from google.colab import auth

try:
    auth.authenticate_user()
    print("Usuário autenticado com sucesso no Google Cloud!")
except Exception as e:
    print(f"[Erro de Autenticação]: {e}")

Usuário autenticado com sucesso no Google Cloud!


In [ ]:
from google.cloud import bigquery
import sys

client = bigquery.Client(project='data-science-brasileirao')
query = """
    SELECT * FROM `data-science-brasileirao.brasileirao_dataset.dados_brazilian_league`
    LIMIT 5
"""

try:
    print("Iniciando a extração dos dados do Campeonato Brasileiro...")
    df_brasileirao = client.query(query).to_dataframe()
    print("Os dados foram carregados com sucesso para o ambiente Python!")
except Exception as e:
    print(f"[Erro na conexão com o BigQuery]: {e}", file=sys.stderr)

Iniciando a extração dos dados do Campeonato Brasileiro...
Os dados foram carregados com sucesso para o ambiente Python!


In [ ]:
df_brasileirao.head()

,season,place,team,points,played,won,draw,loss,goals,goals_taken,goals_diff
0,2011,3,Fluminense,63,38,20,3,15,60,51,9
1,2019,15,Botafogo,43,38,13,4,21,31,45,-14
2,2015,12,Flamengo,49,38,15,4,19,45,53,-8
3,2008,3,Cruzeiro,67,38,21,4,13,59,44,15
4,2007,20,América-RN,17,38,4,5,29,24,80,-56


#O Primeiro desafio técnico ao lidar com dados no BigQuery.

## 1. O Desafio Técnico
Neste projeto, enfrentei um problema comum em projetos de ciência de dados: a **incompatibilidade de formatos**. Os dados brutos do Campeonato Brasileiro estavam armazenados em arquivos `.txt`, um formato que dificulta a criação de tabelas estruturadas diretamente no **Google BigQuery**.

## 2. A Estratégia de Solução
Para resolver isso, estruturei um processo de ETL dividido em duas etapas:
1. **Transformação:** Utilizei Python para ler os arquivos e converter cada linha em um dicionário, exportando-os para o formato `JSON`.
2. **Carga:** Implementei um script utilizando bibliotecas específicas para realizar o upload e a criação automática da tabela no BigQuery.

## 3. Conceitos de Python Aplicados
Durante o desenvolvimento, apliquei conceitos fundamentais aprendidos na formação:
* **Manipulação de Variáveis e Strings:** Para limpar e formatar os dados brutos do texto.
* **Estruturas de Dados (Dicionários e Listas):** Essenciais para organizar a informação antes da conversão para JSON.
* **Tratamento de Exceções (Try/Except):** Para garantir que erros na estrutura do arquivo não interrompessem o processo de carga.

In [ ]:
import json

arquivo_entrada = 'matches-2003-2025.txt'
arquivo_saida = 'matches-2003-2025.json'
cabecalho = ["date", "hour", "home_team", "score", "away_team"]

separador = ';'

def converter_txt_para_jsonlines():
    try:
        linhas_gravadas = 0

        with open(arquivo_entrada, 'r', encoding='latin-1') as txt_file, \
             open(arquivo_saida, 'w', encoding='utf-8') as json_file:

            for linha in txt_file:
                dados = linha.strip().split(separador)

                if dados and len(dados) > 1:
                    jogo_formatado = dict(zip(cabecalho, dados))
                    linha_json = json.dumps(jogo_formatado, ensure_ascii=False)
                    json_file.write(linha_json + '\n')
                    linhas_gravadas += 1

                elif linha.strip() != "":
                    print(f"Aviso: Linha ignorada: {linha.strip()}")

        print(f"Concluído! {linhas_gravadas} jogos gravados no '{arquivo_saida}' (Formato JSON Lines).")

    except FileNotFoundError:
        print(f"Erro: O arquivo '{arquivo_entrada}' não foi encontrado.")

if __name__ == "__main__":
    converter_txt_para_jsonlines()

##Criação da tabela "matches" e "public matches"

Para consolidar os aprendizados obtidos na trilha de Python da Alura, desenvolvi um script que atua na camada de carga (**Load**) do processo de dados.

A lógica central do script utiliza **estruturas de repetição** e **manipulação de dicionários** para processar as informações que foram convertidas de `.txt` para o formato compatível com o **BigQuery**. O script automatiza as seguintes etapas:

* **Definição do Schema:** Utiliza conceitos de **variáveis** e **tipagem de dados** para estruturar como os dados de cada partida serão armazenados de forma consistente.
* **Tratamento de Exceções:** Aplica blocos `try` e `except` para garantir que a criação da tabela não falhe caso encontre dados inconsistentes ou problemas de conexão.
* **Modularização:** O código é organizado em **funções com parâmetros e retornos**, facilitando a manutenção e seguindo boas práticas de documentação com **docstrings** e **Type hints**.

Dessa forma, a solução transforma dados brutos em uma tabela estruturada, pronta para análises avançadas de Ciência de Dados.

In [ ]:
import pandas as pd
from google.cloud import bigquery
import sys

#1. Definição do Schema
#Criamos uma lista de objetos SchemaField.
schema_exemplo = [
    bigquery.SchemaField("date", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("hour", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("home_team", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("score", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("away_team", "STRING", mode="NULLABLE"),
]

def carregar_json_com_schema_seguro():
    """
    Função que carrega dados JSON no BigQuery forçando a validação
    de um schema pré-definido.
    """
    client = bigquery.Client(project="data-science-brasileirao")
    id_tabela = "data-science-brasileirao.brasileirao_dataset.dados_matches"
    caminho_arquivo = "matches-2003-2025.json"

    print("Passo 1: Leitura do arquivo e tratamento das colunas com Pandas...")
    try:
        df = pd.read_json(caminho_arquivo, lines=True, dtype={"hour": str, "score": str})
        df = df.iloc[1:].reset_index(drop=True)
        df = df.astype(str)
        print("Os dados foram carregados com sucesso para o ambiente Python!")

    except Exception as e:
        print(f"Erro ao ler/tratar o arquivo com Pandas: {e}", file=sys.stderr)
        sys.exit(1)

    #2. Configuração do Job
    job_config = bigquery.LoadJobConfig(
        schema=schema_exemplo,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    print("Passo 2: Iniciando o carregamento com validação de Schema...")

    try:
        job = client.load_table_from_dataframe(
            df,
            id_tabela,
            job_config=job_config
            )

        job.result()
        if job.errors:
            print("Atenção! O arquivo foi lido, mas o BigQuery rejeitou os dados.")
            print(f"Detalhes do erro no BigQuery: {job.errors}")
        else:
            print("Sucesso! Os dados foram validados e a tabela foi criada.")

    except Exception as e:
        print(f"Erro Inesperado no Python: {e}")

if __name__ == "__main__":
    carregar_json_com_schema_seguro()

In [ ]:
import pandas as pd
from google.cloud import bigquery
import sys

#1. Definição do Schema
#Criamos uma lista de objetos SchemaField.
schema_exemplo = [
    bigquery.SchemaField("year", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("team", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("avg_attendance", "STRING", mode="NULLABLE"),
]

def carregar_json_com_schema_seguro():
    """
    Função que carrega dados JSON no BigQuery forçando a validação
    de um schema pré-definido.
    """
    client = bigquery.Client(project="data-science-brasileirao")
    id_tabela = "data-science-brasileirao.brasileirao_dataset.dados_public_matches"
    caminho_arquivo = "average-attendance-2003-2019.json"

    print("Passo 1: Leitura do arquivo e tratamento das colunas com Pandas...")
    try:
        df = pd.read_json(caminho_arquivo, lines=True)
        df = df.rename(columns={
            "year": "year",
            "team": "team",
            "avg_attendance": "avg_attendance",
          })
        df = df.astype(str)
        print("Os dados foram carregados com sucesso para o ambiente Python!")

    except Exception as e:
        print(f"Erro ao ler/tratar o arquivo com Pandas: {e}", file=sys.stderr)
        sys.exit(1)

    #2. Configuração do Job
    job_config = bigquery.LoadJobConfig(
        schema=schema_exemplo,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    print("Passo 2: Iniciando o carregamento com validação de Schema...")

    try:
        job = client.load_table_from_dataframe(
            df,
            id_tabela,
            job_config=job_config
            )

        job.result()
        if job.errors:
            print("Atenção! O arquivo foi lido, mas o BigQuery rejeitou os dados.")
            print(f"Detalhes do erro no BigQuery: {job.errors}")
        else:
            print("Sucesso! Os dados foram validados e a tabela foi criada.")

    except Exception as e:
        print(f"Erro Inesperado no Python: {e}")

if __name__ == "__main__":
    carregar_json_com_schema_seguro()

## ⚽ Ingestão e atualização de dados

Este script abaixo tem como objetivo automatizar a atualização do nosso banco de dados no BigQuery utilizando fontes de dados externas.

### 🎯 O que este script faz?
Ele soluciona a ausência de dados nas nossas tabelas principais, inserindo informações atualizadas sobre:
* **Média de Público dos times (2022 a 2025)**
* **Classificação Geral do campeonato (2025)**
* Obs: **Os anos 2020 e 2021 não tiveram público nos estádios devido a Covid-19**

O código lê essas informações a partir de abas distintas de um arquivo Excel local e, em seguida, executa um processo de *append* (adição), inserindo os novos registros diretamente nas tabelas específicas do BigQuery.

### 📚 Aprendizados aplicados
Este projeto consolida os conhecimentos adquiridos na formação de Python

* **Utilização de Bibliotecas:** Importação de pacotes externos (`pandas` e `google-cloud-bigquery`) para expansão das capacidades do código
* **Modularização com Funções:** Criação de funções personalizadas com passagem de parâmetros para separar as responsabilidades de leitura e carregamento
* **Tratamento de Exceções:** Implementação de cláusulas `try` e `except` para capturar erros de arquivo não encontrado ou falhas de conexão, tornando o pipeline resiliente.

### ⚙️ Instruções de execução
Antes de executar a célula de código abaixo, siga estes passos:
1. Faça o **upload do arquivo Excel** na aba de arquivos aqui do Google Colab.
2. Autentique sua conta do Google Cloud para liberar as permissões de escrita no BigQuery.
3. Altere as variáveis de caminho do arquivo e os *IDs* das tabelas destino no código para refletir os nomes reais do seu projeto.

In [ ]:
import pandas as pd
from google.cloud import bigquery
import sys

def ler_dados_excel(caminho_arquivo: str, nome_aba: str):
    """
    Função para ler uma aba específica do Excel.
    """
    try:
        df = pd.read_excel(caminho_arquivo, sheet_name=nome_aba)
        print(f"Os dados da aba '{nome_aba}' lidos com sucesso!")
        return df
    except FileNotFoundError:
        print(f"[Erro]: O arquivo {caminho_arquivo} não foi encontrado.", file=sys.stderr)
    except Exception as e:
        print(f"[Erro Inesperado na leitura]: {e}", file=sys.stderr)

def carregar_dados_bigquery(dataframe, id_tabela_destino: str):
    """
    Função para enviar os dados para o BigQuery.
    """
    try:
        id_do_projeto = 'data-science-brasileirao'
        client = bigquery.Client(project=id_do_projeto)
        job_config = bigquery.LoadJobConfig(
            write_disposition="WRITE_APPEND"
        )


        job = client.load_table_from_dataframe(dataframe, id_tabela_destino, job_config=job_config)
        job.result()

        print(f"Dados carregados com sucesso na tabela {id_tabela_destino}.")
    except Exception as e:
        print(f"[Erro ao carregar no BigQuery]: {e}", file=sys.stderr)

def main():
    print("--- Atualização de dados do Brasileirão ---")
    caminho_excel = "dados_adicionais_br.xlsx"

    #Processar dados de Público
    df_publico = ler_dados_excel(caminho_excel, "publico")
    if df_publico is not None:
        df_publico = df_publico.astype({
            "year": str,
            "team": str,
            "avg_attendance": str
        })
        carregar_dados_bigquery(df_publico, "data-science-brasileirao.brasileirao_dataset.dados_public_matches")

    #Processar dados de Classificação
    df_classificacao = ler_dados_excel(caminho_excel, "classificacao")
    if df_classificacao is not None:
        carregar_dados_bigquery(df_classificacao, "data-science-brasileirao.brasileirao_dataset.dados_brazilian_league")

if __name__ == "__main__":
    main()

## ⚽ Iniciando a criação dos dataframes

In [ ]:
client = bigquery.Client(project='data-science-brasileirao')
query = """
    SELECT * FROM `data-science-brasileirao.brasileirao_dataset.dados_brazilian_league`
"""

try:
    print("Iniciando a extração dos dados do Campeonato Brasileiro...")
    df_brasileirao = client.query(query).to_dataframe()
    print("Os dados foram carregados com sucesso para o ambiente Python!")
except Exception as e:
    print(f"[Erro na conexão com o BigQuery]: {e}", file=sys.stderr)

In [ ]:
client = bigquery.Client(project='data-science-brasileirao')
query = """
    SELECT * FROM `data-science-brasileirao.brasileirao_dataset.dados_matches`
"""

try:
    print("Iniciando a extração dos dados do Campeonato Brasileiro...")
    df_partidas = client.query(query).to_dataframe()
    print("Os dados foram carregados com sucesso para o ambiente Python!")
except Exception as e:
    print(f"[Erro na conexão com o BigQuery]: {e}", file=sys.stderr)

In [ ]:
import pandas as pd
df_partidas = pd.DataFrame(df_partidas)

df_partidas [['home_team_score', 'away_team_score']] = df_partidas['score'].str.split('x', expand=True)
df_partidas

In [ ]:
colunas_placar = ['home_team_score', 'away_team_score']
df_partidas[colunas_placar] = df_partidas[colunas_placar].apply(pd.to_numeric, errors='coerce').astype(int)

In [ ]:
df_partidas['date'] = pd.to_datetime(df_partidas['date'], format='%d/%m/%Y')
df_partidas

In [ ]:
soma_gols = df_partidas['home_team_score'] + df_partidas['away_team_score']
df_partidas['total_gols'] = soma_gols
df_partidas

In [ ]:
client = bigquery.Client(project='data-science-brasileirao')
query = """
    SELECT * FROM `data-science-brasileirao.brasileirao_dataset.dados_public_matches`
"""

try:
    print("Iniciando a extração dos dados do Campeonato Brasileiro...")
    df_media_publico = client.query(query).to_dataframe()
    print("Os dados foram carregados com sucesso para o ambiente Python!")
except Exception as e:
    print(f"[Erro na conexão com o BigQuery]: {e}", file=sys.stderr)

In [ ]:
colunas_media = ['year', 'avg_attendance']
df_media_publico[colunas_media] = df_media_publico[colunas_media].apply(pd.to_numeric, errors='coerce').astype(int)

In [ ]:
client = bigquery.Client(project='data-science-brasileirao')
query = """
    SELECT * FROM `data-science-brasileirao.brasileirao_dataset.dados_teams`
"""

try:
    print("Iniciando a extração dos dados do Campeonato Brasileiro...")
    df_times = client.query(query).to_dataframe()
    print("Os dados foram carregados com sucesso para o ambiente Python!")
except Exception as e:
    print(f"[Erro na conexão com o BigQuery]: {e}", file=sys.stderr)

## ⚽ EDA Brasileirão

### 1. Quais as 3 edições com maior número de gols?

Para iniciar nossa Análise Exploratória de Dados (EDA), vamos investigar o volume ofensivo do campeonato ao longo do tempo. O objetivo desta etapa é descobrir quais foram os anos mais emocionantes em termos de bolas na rede.

**Abordagem técnica:** Utilizaremos a biblioteca `pandas` para agrupar os dados pela coluna de edição/ano e realizaremos a soma total de gols (mandantes + visitantes). Em seguida, ordenaremos os resultados de forma decrescente para extrair o "Top 3".

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
top_3_edicoes_gols = df_brasileirao.groupby('season')['goals'].sum().nlargest(3)

print("As 3 edições com maior número de gols foram: ")
print(top_3_edicoes_gols)

### 2. Quais as 3 equipes com o maior número de gols?

Agora que sabemos quais edições foram mais artilheiras, vamos focar nos protagonistas: os times. Queremos identificar quais são as equipes mais letais da história deste dataset.

**Abordagem técnica:** Semelhante ao passo anterior, vamos agrupar nosso DataFrame, mas desta vez utilizando a coluna correspondente aos nomes dos times. Calcularemos a soma acumulada de gols de cada equipe em toda a base histórica e selecionaremos as três primeiras posições.

In [ ]:
top_3_equipes_gols = df_brasileirao.groupby('team')['goals'].sum().nlargest(3)

print("As 3 equipes com o maior número de gols foram: ")
print(top_3_equipes_gols)

### 3. Quais as 3 equipes com a maior média de pontos em todas as edições?

Fazer muitos gols é importante, mas a consistência é o que define as melhores equipes. Nesta etapa, vamos analisar o aproveitamento histórico dos times, calculando a média de pontos que eles conquistam. Isso nos ajuda a entender quais times mantêm um alto nível de competitividade, independentemente da quantidade de jogos ou edições disputadas.

**Abordagem técnica:** Vamos agrupar os dados pelos times e utilizar uma função de agregação de média (`mean()`) sobre a coluna de pontos. *Nota de atenção:* É importante garantir que apenas times que participaram de um número mínimo aceitável de jogos sejam considerados, para evitar distorções de times que jogaram pouco e tiveram 100% de aproveitamento.

In [ ]:
top_3_media_pontos = df_brasileirao.groupby('team')['points'].mean().nlargest(3)

print("As 3 equipes com a maior média de pontos em todas as edições foram: ")
print(top_3_media_pontos)

### 💡 Insight Analítico: Tratamento de Viés de Amostra

Durante a análise da média de pontos (Perguntas 3 e 5), percebi que times com apenas uma participação, como o Mirassol, distorciam o ranking. O baixo volume de dados criava um **viés de amostra pequena**, mascarando a real performance histórica das equipes.

Para trazer uma análise mais justa e focada na consistência estatística, apliquei uma técnica de filtragem (*threshold*), considerando apenas equipes com **mais de 3 temporadas/jogos** na história da competição.

> O resultado mudou completamente o cenário e nos revelou os verdadeiros líderes em consistência!

In [ ]:
estatisticas_times = df_brasileirao.groupby('team')['points'].agg(['mean', 'count'])
minimo_temporadas = 3

times_qualificados = estatisticas_times[estatisticas_times['count'] >= minimo_temporadas]

top_3_media_pontos_v1 = times_qualificados['mean'].nlargest(3).round(1)

print("As 3 equipes com a maior média de pontos em todas as edições foram: ")
print(top_3_media_pontos_v1)

### 4. Quais as 3 maiores médias de público e em quais anos ocorreram?

O futebol não é feito apenas pelo que acontece dentro de campo, mas também pelo engajamento nas arquibancadas. Vamos descobrir em quais edições os torcedores estiveram mais presentes nos estádios.

**Abordagem técnica:** Realizaremos um agrupamento pelas edições (anos) do campeonato e calcularemos a média aritmética da coluna de público pagante/presente. O resultado nos mostrará o Top 3 anos com maior adesão média da torcida.

In [ ]:
top_3_media_publico_por_ano = df_media_publico.groupby('year')['avg_attendance'].mean().nlargest(3)

print("Os 3 anos com a maior média de público foram: ")
print(top_3_media_publico_por_ano)

### 5. Quais os 3 times com a maior média e os 3 times com a pior média de performance?

Para finalizar nossa exploração inicial, vamos olhar para os extremos do campeonato. Entender quais times dominam as estatísticas (Top 3) e quais têm os maiores desafios nos dá uma visão completa da disparidade técnica ao longo dos anos.

*(Avaliando a média de pontos conquistados por partida)*

**Abordagem técnica:** Aproveitando o cálculo de média de pontos da questão 3, vamos extrair os 3 primeiros colocados (`nlargest()`) e os 3 últimos colocados (`nsmallest()`). Visualizar esses extremos nos ajuda a consolidar o panorama geral do torneio.

In [ ]:
media_pontos_todos_times = df_brasileirao.groupby('team')['points'].mean()

melhores_3 = media_pontos_todos_times.nlargest(3)
piores_3 = media_pontos_todos_times.nsmallest(3)

print("Top 3 - Maior Média de Pontos: ")
print(melhores_3)
print("/---------------------------------/")
print("Top 3 - Pior Média de Pontos: ")
print(piores_3)
print("/---------------------------------/")

In [ ]:
estatisticas_times = df_brasileirao.groupby('team')['points'].agg(['mean', 'count'])
minimo_temporadas = 3

times_qualificados = estatisticas_times[estatisticas_times['count'] >= minimo_temporadas]

media_pontos_todos_times = times_qualificados.mean()

melhores_3 = times_qualificados['mean'].nlargest(3).round(1)
piores_3 = times_qualificados['mean'].nsmallest(3).round(1)

print("Top 3 - Maior Média de Pontos: ")
print(melhores_3)
print("/---------------------------------/")
print("Top 3 - Pior Média de Pontos: ")
print(piores_3)
print("/---------------------------------/")